In [ ]:
#training, testing, and fine-tuning BERT
!pip install -q datasets transformers
!pip install -q evaluate

from datasets import load_dataset

dataset = load_dataset("simlab-vs/meajor_cleaned_preprocessed")
dataset
print(dataset["train"].column_names)
dataset["train"][0]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00


README.md:   0%|          | 0.00/2.63k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 82.7MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/108685 [00:00<?, ? examples/s]

['sender', 'sender_domain', 'receiver', 'receiver_domain', 'date', 'subject', 'content_types', 'body', 'urls', 'url_count', 'url_length_max', 'url_length_avg', 'url_subdom_max', 'url_subdom_avg', 'attachment_count', 'has_attachments', 'attachment_types', 'language', 'source', 'label']


{'sender': 'd66e9e64b006d6bca649f1c945129c42c43836872b2eadfb822b6af04b3212cb',
 'sender_domain': 'enron.com',
 'receiver': '35c5a9fb9fba3b8737ed7cef2a87e427a73db4fca85f6be4dc6a28b031887a0d',
 'receiver_domain': 'enron.com',
 'date': '2001-06-29 09:37:04-05:00',
 'subject': '[ORGANIZATION] failover plan.',
 'content_types': 'text/plain',
 'body': 'Hi [NAME],  \n\nTonight we are rolling out a new report.  Currently, only you and [NAME] have access to it.  You\'ll select it off the main reports menu.  It will say "[ORGANIZATION] Download report".  When you launch it, it will show the data on screen like all other reports.  There is a button at the bottom that says "Export".  Clicking it will give you several selections.  Choose the one that says "Save to [ORGANIZATION] XML".  You can then save it as a file which can be mailed to [ORGANIZATION].  \n\nLet me know if there is anyone else who needs access.\n\nThanks!\n[NAME]',
 'urls': None,
 'url_count': 0.0,
 'url_length_max': 0.0,
 'url_le

In [ ]:
# to clean and filter - will be droping any nulls, not english words and unlabeled rows
df = dataset["train"].to_pandas()

df = df.dropna(subset=["label", "body"])

df = df[df["language"] == "en"]

df['label'] = df["label"].astype(int)

print(df["label"].value_counts())
print(len(df))



label
0    56082
1    26093
Name: count, dtype: int64
82175


In [ ]:
#features
df["subject"] = df["subject"].fillna("")
df["body"] = df['body'].fillna("")
df["sender_domain_length"] = df['sender_domain'].fillna("").str.len()
df["subject_length"] = df['subject'].str.len()
df["body_length"] = df['body'].str.len()

##
def pct_punctuation(text):
    if len(text) == 0:
        return 0.0
    punct_count = sum(1 for c in text if c in "!?$%*#@")
    return punct_count / len(text)

df["precentage_punctuation_marks"] = df["body"].apply(pct_punctuation)

df["text"] = df['subject'] + " " + df['body']

In [ ]:
# training and testing - tokenizing

from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer

train_df, test_df = train_test_split(df,test_size = 0.2, stratify = df['label'], random_state = 42)
val_df,test_df = train_test_split(test_df,test_size = 0.5, stratify = test_df['label'], random_state = 42)

Model_Name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(Model_Name)

def to_hf_dataset(pdf):
  return Dataset.from_pandas(pdf)

train_dataset = to_hf_dataset(train_df)
val_dataset = to_hf_dataset(val_df)
test_dataset = to_hf_dataset(test_df)

def tokenize_fn(batch):
  return tokenizer(batch["text"], truncation=True)

train_dataset = train_dataset.map(tokenize_fn, batched=True)
val_dataset = val_dataset.map(tokenize_fn, batched=True)
test_dataset = test_dataset.map(tokenize_fn, batched=True)


Map:   0%|          | 0/65740 [00:00<?, ? examples/s]

Map:   0%|          | 0/8217 [00:00<?, ? examples/s]

Map:   0%|          | 0/8218 [00:00<?, ? examples/s]

In [ ]:
# fine-tuning model
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

import evaluate
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(Model_Name, num_labels=2)

accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "precision": precision.compute(predictions=predictions, references=labels)["precision"],
        "recall": recall.compute(predictions=predictions, references=labels)["recall"],
        "f1": f1.compute(predictions=predictions, references=labels)["f1"]
    }
args = TrainingArguments(
    output_dir="./bert_phising",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate= 2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16 = True,
    dataloader_num_workers = 2,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.038609,0.040236,0.988560,0.979779,0.984285,0.982027
2,0.018283,0.038171,0.990507,0.988421,0.981602,0.985000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4110, training_loss=0.044842659614962095, metrics={'train_runtime': 1658.4226, 'train_samples_per_second': 79.28, 'train_steps_per_second': 2.478, 'total_flos': 1.7415522018153984e+16, 'train_loss': 0.044842659614962095, 'epoch': 2.0})

In [ ]:
results = trainer.evaluate(test_dataset)
print(results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.017033,0.043948,2,0.989048,0.984615,0.980843,0.982726


{'eval_loss': 0.043947964906692505, 'eval_accuracy': 0.9890484302750061, 'eval_precision': 0.9846153846153847, 'eval_recall': 0.9808429118773946, 'eval_f1': 0.982725527831094}


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

In [ ]:
# save the trained DistilBERT model
trainer.save_model("distilbert_phishing_model")
tokenizer.save_pretrained("distilbert_phishing_model")

print("Model saved successfully")

NameError: name 'trainer' is not defined